# الدرس 11 - بروتوكول الوكيل إلى الوكيل (A2A)


## الإعداد


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ما هو بروتوكول A2A؟

يُعتبر **بروتوكول Agent-to-Agent (A2A)** معيارًا مفتوحًا يتيح لوكلاء الذكاء الاصطناعي التواصل،
اكتشاف بعضهم البعض، والتعاون — حتى عندما يُبنى كلٌ منهم على أُطر عمل مختلفة أو يُستضاف
من قبل خدمات مختلفة.

المفاهيم الرئيسية:

- **الاكتشاف** – ينشر الوكلاء *بطاقة الوكيل* التي تصف قدراتهم، مما يجعل من
  السهل على الوكلاء الآخرين (أو المنسقين) العثور على المختص المناسب لمهمة.
- **تمرير الرسائل** – يتبادل الوكلاء رسائل منظمة عبر بروتوكول مشترك، بحيث
  يمكن فهم طلب من وكيل واحد وتنفيذه بواسطة وكيل آخر بغض النظر عن
  تنفيذاته الداخلية.
- **دورة حياة المهمة** – يعرّف A2A حالات مثل *مُقدّم*, *قيد المعالجة*, *مُكتمل*, و
  *فاشل*, مما يمنح المنسق رؤية كاملة حول كيفية تقدم مهمة مُفوَّضة.

في هذا الدرس نقوم بمحاكاة التعاون على نمط A2A من خلال توصيل ثلاثة وكلاء سفر متخصصين
إلى سير عمل حيث يساهم كل وكيل بخبرته وينقل النتائج إلى التالي.


## إنشاء وكلاء سفر متخصصين


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## التعاون متعدد الوكلاء عبر سير العمل

نقوم بربط الوكلاء الثلاثة في سير عمل متسلسل يعكس تمرير الرسائل A2A:

1. **CurrencyExchangeAgent** يستقبل طلب المستخدم ويُقدّم إرشادات بشأن العملة.
2. **ActivityPlannerAgent** يستقبل السياق المحسّن ويضيف توصيات بالأنشطة.
3. **TravelManagerAgent** يصوغ المدخلين في موجز سفر نهائي.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## فهم A2A في بيئة الإنتاج

في بيئة الإنتاج يفتح بروتوكول A2A سيناريوهات قوية عبر الخدمات:

| Capability | Description |
|---|---|
| **التشغيل البيني بين أطر العمل** | يمكن لوكيل بُني باستخدام إطار عمل واحد تفويض المهام إلى وكيل بُني باستخدام أي إطار عمل آخر متوافق مع A2A، مما يتيح تفاعلًا حقيقيًا بين المنظمات. |
| **حدود الخدمة** | يمكن للوكلاء أن يتواجدوا في خدمات مصغرة منفصلة، أو مناطق سحابية مختلفة، أو حتى مؤسسات مختلفة بينما يظلون يتعاونون بسلاسة. |
| **الاكتشاف الديناميكي** | يمكن لمنسق المهام الاستعلام من سجل Agent Card في وقت التشغيل للعثور على المتخصص الأنسب لمهمة فرعية معينة. |
| **البث & الإشعارات الدفعية** | يدعم A2A أحداث مرسلة من الخادم (SSE) لتحديثات التقدم في الوقت الحقيقي والإشعارات الدفعية للمهام طويلة الأمد. |

سير العمل الذي أنشأنا أعلاه هو نسخة مبسطة، داخلية تعمل ضمن نفس العملية من هذا النمط. في نشر حقيقي
سيكشف كل وكيل عن نقطة نهاية HTTP، وينشر Agent Card، ويتواصل
عبر بروتوكول A2A JSON-RPC.


## الملخص

في هذا الدرس تعلمت:

1. **ما هو بروتوكول A2A** — معيار مفتوح لاكتشاف بين الوكلاء، والمراسلة،
   وإدارة المهام.
2. **كيفية إنشاء وكلاء متخصّصين** — وكيل صرف العملات، ووكيل مخطط الأنشطة،
   ومنسق Travel Manager.
3. **كيفية ربط الوكلاء في سير عمل** — باستخدام `WorkflowBuilder` لنمذجة تمرير
   الرسائل بالتسلسل بين الوكلاء.
4. **كيف يعمل A2A في بيئة الإنتاج** — تمكين التعاون عبر الأطر وعبر الخدمات
   مع الاكتشاف الديناميكي والتحديثات بالتدفق.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
إخلاء المسؤولية:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية Co-op Translator (https://github.com/Azure/co-op-translator). بينما نسعى لتحقيق الدقة، يرجى ملاحظة أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر المعتمد. بالنسبة للمعلومات الحرجة، يُنصح بالاعتماد على ترجمة بشرية محترفة. لا نتحمّل أي مسؤولية عن أي سوء فهم أو تفسيرات خاطئة ناتجة عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
